In [10]:
# Install dependencies using uv (recommended for speed) or pip
!uv pip install -q claude-agent-sdk rich python-dotenv || pip install -q claude-agent-sdk rich python-dotenv


In [12]:
import os
import asyncio
from dotenv import load_dotenv
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage
from rich.console import Console
from rich.markdown import Markdown

print("SDK and utility imports successful.")


SDK and utility imports successful.


In [13]:
load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    print("WARNING: ANTHROPIC_API_KEY environment variable is missing.")
    print("Please set ANTHROPIC_API_KEY in your environment or .env file before making query() calls.")
else:
    masked = api_key[:7] + "..." + api_key[-4:] if len(api_key) > 11 else "***"
    print(f"Anthropic API key successfully loaded: {masked}")


Anthropic API key successfully loaded: sk-ant-...jQAA


In [14]:
TARGET_DIR = "data"  # Path to target codebase directory

# Define execution options and whitelist read-only tools
options = ClaudeAgentOptions(
    allowed_tools=["Read", "Glob", "Grep"],
    model="claude-haiku-4-5-20251001",
)

console = Console()
console.print("[bold green]Agent options initialized.[/bold green]")
console.print(f"Target Directory: [cyan]{TARGET_DIR}[/cyan]")
console.print(f"Allowed tools whitelist: [yellow]{options.allowed_tools}[/yellow]")


Agent options initialized.

Target Directory: data

Allowed tools whitelist: ['Read', 'Glob', 'Grep']

In [15]:
TASK = f"""
Scan the codebase located at '{TARGET_DIR}' and identify all TODO and FIXME comments.

For each match, detail:
- Relative file path
- Line number
- Comment text
- Brief summary of the task or issue described

Organize output into a clean markdown document grouped by file.
"""

print("Task prompt defined:")
print("-" * 50)
print(TASK.strip())
print("-" * 50)


Task prompt defined:
--------------------------------------------------
Scan the codebase located at 'data' and identify all TODO and FIXME comments.

For each match, detail:
- Relative file path
- Line number
- Comment text
- Brief summary of the task or issue described

Organize output into a clean markdown document grouped by file.
--------------------------------------------------


In [16]:
async def execute_audit(task_prompt: str, agent_options: ClaudeAgentOptions) -> str:
    final_output = ""
    console.print("[bold blue]Starting agent loop via query()...[/bold blue]")

    async for message in query(prompt=task_prompt, options=agent_options):
        # Live progress: print what Claude is doing each turn
        if isinstance(message, AssistantMessage):
            tool_calls = [
                block.name
                for block in message.content
                if hasattr(block, "type") and block.type == "tool_use"
            ]
            if tool_calls:
                console.print(f"[dim]  → Tool calls requested: {', '.join(tool_calls)}[/dim]")

        # Final result: check subtype before accessing .result
        if isinstance(message, ResultMessage):
            if message.subtype == "success":
                # .result is only present on the success variant
                final_output = message.result or ""
                console.print("[bold green]✓ Execution completed successfully.[/bold green]")
            else:
                final_output = f"Execution stopped with status: {message.subtype}"
                console.print(f"[bold red]✗ Execution stopped: {message.subtype}[/bold red]")

    return final_output


# Execute the asynchronous query loop in a worker thread with subprocess support
try:
    # In Jupyter, await the async function directly (avoid asyncio.run in a worker thread)
    response_text = await execute_audit(TASK, options)
except Exception as e:
    response_text = f"Execution failed: {e}"
    console.print(f"[bold red]✗ Execution failed: {e}[/bold red]")

console.print("\n[bold cyan]--- Agent Response ---[/bold cyan]\n")
console.print(Markdown(response_text))

Starting agent loop via query()...

✗ Execution failed: Failed to start Claude Code: 

--- Agent Response ---

Execution failed: Failed to start Claude Code:

In [17]:
REPORT_PATH = "todo_fixme_report.md"

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("# Codebase TODO / FIXME Audit Report\n\n")
    f.write(f"Target directory: `{TARGET_DIR}`\n\n")
    f.write(response_text)

console.print(f"[bold green]Audit report written to {REPORT_PATH}[/bold green]")

Audit report written to todo_fixme_report.md